In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re

## Apartments

In [ ]:
apartments = pd.read_csv('data/raw/apartments.csv')

apartments['full_cost'] = apartments['price'] + apartments['charges']
apartments['disposition_clean'] = (
    apartments['disposition']
    .str.replace('DISP_', '', regex=False)
    .str.replace('_', '+')
)
apartments['district'] = apartments['district'].str.replace('Praha-', '', regex=False)
apartments['price_per_m2'] = apartments['full_cost'] / apartments['surface']

# remove undefined dispositions
apartments['disposition_clean'] = apartments['disposition_clean'].replace(['UNDEFINED', 'OSTATNI'], np.nan)

# drop duplicates
apartments = apartments.drop_duplicates()

# remove extreme sizes
apartments = apartments[(apartments['surface'] > 10) & (apartments['surface'] <= 250)]

# cap at 95th percentile — removes luxury outliers beyond student affordability
apartments = apartments[apartments['full_cost'] < apartments['full_cost'].quantile(0.95)]

print(apartments.shape)
apartments.head()

In [ ]:
center = [
    'Staré Město', 'Nové Město', 'Malá Strana'
]

inner_city = [
    'Vinohrady', 'Žižkov', 'Karlín', 'Smíchov',
    'Holešovice', 'Vršovice', 'Nusle', 'Dejvice',
    'Bubeneč', 'Podolí', 'Libeň', 'Vysočany',
    'Košíře', 'Břevnov', 'Střešovice', 'Vokovice',
    'Radlice', 'Jinonice', 'Hlubočepy', 'Braník',
    'Michle', 'Strašnice', 'Kobylisy', 'Prosek',
]

outer_city = [
    'Stodůlky', 'Chodov', 'Letňany', 'Krč',
    'Modřany', 'Záběhlice', 'Kyje', 'Kamýk',
    'Bohnice', 'Hostivař', 'Černý Most', 'Řepy',
    'Háje', 'Horní Měcholupy', 'Malešice', 'Troja',
    'Liboc', 'Lhotka', 'Štěrboholy', 'Horní Počernice',
    'Čimice', 'Libuš', 'Kunratice', 'Dolní Chabry',
    'Ďáblice', 'Hostavice', 'Dolní Počernice', 'Komořany',
    'Běchovice', 'Dubeč', 'Řeporyje',
]

def classify_zone(district):
    if district in center:       return 'center'
    elif district in inner_city: return 'inner_city'
    else:                        return 'outer_or_suburbs'

apartments['zone'] = apartments['district'].apply(classify_zone)

district_stats = apartments.groupby(['zone', 'disposition_clean']).agg(
    avg_rent=('full_cost', 'mean'),
    avg_m2_price=('price_per_m2', 'mean'),
    count=('full_cost', 'size')
).reset_index()

district_stats

In [ ]:
keep = ['GARSONIERA', '1+KK', '1+1', '2+KK', '2+1', '3+KK', '3+1', '4+KK', '4+1']

apt_stats = (
    apartments[apartments['disposition_clean'].isin(keep)]
    .groupby(['zone', 'disposition_clean'])
    .agg(avg_rent=('full_cost', 'mean'), n=('full_cost', 'count'))
    .reset_index()
)

apt_stats

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for zone, grp in apt_stats.groupby('zone'):
    ax.plot(grp['disposition_clean'], grp['avg_rent'], marker='o', label=zone)
ax.set_xlabel('Disposition')
ax.set_ylabel('Avg monthly rent (Kč)')
ax.set_title('Average rent by disposition and zone')
ax.legend()
plt.tight_layout()
plt.show()

## Groceries

In [ ]:
# weekly target quantities per item
# unit matches the target_unit in ITEMS (g, ml, piece)
WEEKLY_QTY = {
    # standard + shared
    'ovesne_vlocky':    500,    # g
    'mleko':            2000,   # ml  (2L)
    'banany':           1000,   # g   (~6 pieces)
    'chleb':            500,    # g   (~1 loaf)
    'maslo':            250,    # g
    'vejce':            10,     # pieces
    'spagety':          500,    # g
    'pesto':            190,    # g   (1 jar)
    'ryze':             500,    # g
    'mrazena_zelenina': 400,    # g
    'kureci_prsa':      500,    # g
    'tofu':             400,    # g
    'cibule':           500,    # g
    'cesnek':           80,     # g   (~1 head)
    'syr_eidam':        200,    # g
    'tunak':            370,    # g   (2x185g tins)
    'jogurt':           300,    # g   (2x150g)
    'kava':             250,    # g   (lasts longer but priced per pack)
    # vegan-only
    'ovesny_napoj':     1000,   # ml  (1L)
    'sojovy_napoj':     1000,   # ml  (1L)
    'repkovy_olej':     200,    # ml  (weekly portion of 1L bottle)
    'cervena_cocka':    500,    # g
    'rajcatova_omacka': 350,    # g   (1 jar)
    'paprika':          300,    # g   (~1-2 pieces)
}

BASKET_STANDARD = {
    'ovesne_vlocky', 'mleko', 'banany', 'chleb', 'maslo', 'vejce',
    'spagety', 'pesto', 'ryze', 'mrazena_zelenina', 'kureci_prsa',
    'tofu', 'cibule', 'cesnek', 'syr_eidam', 'tunak', 'jogurt', 'kava',
}

BASKET_VEGAN = {
    'ovesne_vlocky', 'banany', 'chleb', 'spagety', 'ryze',
    'mrazena_zelenina', 'tofu', 'cibule', 'cesnek', 'kava',
    'ovesny_napoj', 'sojovy_napoj', 'repkovy_olej',
    'cervena_cocka', 'rajcatova_omacka', 'paprika',
}

In [ ]:
def parse_qty(text):
    """Parse quantity and unit from packaging string. Returns (qty, unit) or (None, None)."""
    if not text or pd.isna(text):
        return None, None
    t = str(text).lower().strip()
    if m := re.search(r'(\d+)\s*ks', t):                return int(m.group(1)), 'piece'
    if m := re.search(r'(\d+[,.]?\d*)\s*kg', t):        return float(m.group(1).replace(',', '.')) * 1000, 'g'
    if m := re.search(r'(\d+[,.]?\d*)\s*g(?!\w)', t):  return float(m.group(1).replace(',', '.')), 'g'
    if m := re.search(r'(\d+[,.]?\d*)\s*l(?!\w)', t):  return float(m.group(1).replace(',', '.')) * 1000, 'ml'
    if m := re.search(r'(\d+)\s*ml', t):                return int(m.group(1)), 'ml'
    return None, None


def weekly_cost(row):
    """Convert scraped price to weekly cost based on target quantity."""
    item = row['item']
    price = row['price_czk']
    if pd.isna(price):
        return np.nan
    target_qty = WEEKLY_QTY.get(item)
    if target_qty is None:
        return np.nan
    pack_qty, pack_unit = parse_qty(row.get('packaging', ''))
    # if we can parse the pack size, scale price to target quantity
    if pack_qty and pack_qty > 0:
        return price / pack_qty * target_qty
    # fallback: use price as-is (assume pack ≈ weekly qty)
    return price


def load_store(name):
    path = f'data/rohlik_prices.csv' if name == 'rohlik' else f'data/{name}_prices.csv'
    df = pd.read_csv(path)
    df['weekly_cost'] = df.apply(weekly_cost, axis=1)
    return df[['item', 'title', 'price_czk', 'packaging', 'weekly_cost']]

rohlik = load_store('rohlik')
kosik  = load_store('kosik')
billa  = load_store('billa')
lidl   = load_store('lidl')

rohlik.head()

In [ ]:
# merge all stores into one table
prices = (
    rohlik[['item', 'title', 'weekly_cost']].rename(columns={'title': 'title_rohlik', 'weekly_cost': 'rohlik'})
    .merge(kosik[['item', 'weekly_cost']].rename(columns={'weekly_cost': 'kosik'}), on='item', how='outer')
    .merge(billa[['item', 'weekly_cost']].rename(columns={'weekly_cost': 'billa'}), on='item', how='outer')
    .merge(lidl[['item', 'weekly_cost']].rename(columns={'weekly_cost': 'lidl'}),   on='item', how='outer')
)

prices['online_avg']   = prices[['rohlik', 'kosik']].mean(axis=1)
prices['physical_avg'] = prices[['billa', 'lidl']].mean(axis=1)

prices

In [ ]:
def basket_cost(prices_df, basket_items):
    """Weekly and monthly basket cost from a prices dataframe."""
    sub = prices_df[prices_df['item'].isin(basket_items)].copy()
    weekly_online   = sub['online_avg'].sum()
    weekly_physical = sub['physical_avg'].sum()
    return pd.Series({
        'weekly_online':    round(weekly_online, 1),
        'weekly_physical':  round(weekly_physical, 1),
        'monthly_online':   round(weekly_online * 4.33, 1),
        'monthly_physical': round(weekly_physical * 4.33, 1),
    })

standard_costs = basket_cost(prices, BASKET_STANDARD)
vegan_costs    = basket_cost(prices, BASKET_VEGAN)

summary = pd.DataFrame({'standard': standard_costs, 'vegan': vegan_costs})
print(summary)

In [ ]:
# per-item breakdown for standard basket
standard_breakdown = prices[prices['item'].isin(BASKET_STANDARD)][['item', 'rohlik', 'kosik', 'billa', 'lidl', 'online_avg', 'physical_avg']].copy()
standard_breakdown = standard_breakdown.sort_values('online_avg', ascending=False)

# quick bar chart — most expensive items
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(standard_breakdown))
ax.bar([i - 0.2 for i in x], standard_breakdown['online_avg'],   width=0.4, label='Online avg',   color='#4A7C72')
ax.bar([i + 0.2 for i in x], standard_breakdown['physical_avg'], width=0.4, label='Physical avg', color='#C0513A')
ax.set_xticks(list(x))
ax.set_xticklabels(standard_breakdown['item'], rotation=45, ha='right')
ax.set_ylabel('Weekly cost (Kč)')
ax.set_title('Standard basket — weekly cost per item')
ax.legend()
plt.tight_layout()
plt.show()

## Dorms

In [ ]:
dorms = pd.read_csv('data/raw/dorm_prices.csv')

dorms['beds'] = dorms['room_type'].str.split('-').str[0].astype(int)
dorms['facilities'] = dorms['room_type'].str.contains('se soc. zař.', case=False, na=False).map({True: 'Yes', False: 'No'})
dorms['monthly_cost'] = dorms['price'] * 30

dorm_stats = (
    dorms.groupby(['beds', 'facilities'])
    .agg(avg_cost=('monthly_cost', 'mean'), count=('monthly_cost', 'size'))
    .reset_index()
)

dorm_stats

## Canteen

In [ ]:
canteen = pd.read_csv('data/raw/menza_prices.csv')

main_mean = canteen[canteen['meal_type'] == 'Hlavní jídlo']['price'].mean()
soup_mean = canteen[canteen['meal_type'] == 'Polévka']['price'].mean()
daily_cost = main_mean + soup_mean

canteen_costs = pd.DataFrame({'days_per_week': range(1, 6)})
canteen_costs['monthly_cost'] = canteen_costs['days_per_week'] * 4.33 * daily_cost

print(f'avg main meal: {main_mean:.1f} Kč, avg soup: {soup_mean:.1f} Kč')
canteen_costs

## Save clean data